In [3]:
import numpy as np
import pandas as pd 
import json
import re
from itertools import product
import matplotlib.pyplot as plt
import umap
import inspect
import joblib
from copy import deepcopy
from IPython.display import display
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from solution_utils import build_tables, get_users_text, vectorize_topic_texts, SEED, find_duplicate_groups, read_dialogs, get_groups, calculate_metrics, save_model, MyModelTopics, init_reaction_models, predict_reaction
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score, v_measure_score, silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from scipy.sparse.csgraph import connected_components
from sklearn.metrics.cluster import contingency_matrix
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

/Users/var/Documents/projects/meow/HW/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Тест реакции

In [4]:
hidden_dialogs = read_dialogs('hidden_test.jsonl')

with open('hidden_labels.jsonl', encoding='utf-8') as file:
    records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

reaction_labels = {
    record['dialog_id']: record['reaction']
    for record in records
    if record.get('reaction') is not None
}

hidden_dialogs = [
    dialog
    for dialog in hidden_dialogs
    if dialog['dialog_id'] in reaction_labels
]

hidden_ids = [dialog['dialog_id'] for dialog in hidden_dialogs]

y_hidden_reaction = np.array([
    reaction_labels[dialog_id]
    for dialog_id in hidden_ids
])

REACTION_CLASSES = ['negative', 'neutral', 'positive']

print('Диалогов для оценки:', len(hidden_dialogs))
print('\nРаспределение реакций:')
print(pd.Series(y_hidden_reaction).value_counts())

reaction_models = init_reaction_models()


reaction_predictions = {}
reaction_metrics = {}

for name, model in reaction_models.items():
    predictions = predict_reaction(model, hidden_dialogs)

    reaction_predictions[name] = predictions
    reaction_metrics[name] = calculate_metrics(
        y_hidden_reaction,
        predictions,
        labels=REACTION_CLASSES,
    )


metric_names = [
    'accuracy',
    'balanced_accuracy',
    'macro-f1',
    'weighted-f1',
]

reaction_results = pd.DataFrame({
    name: {
        metric: metrics[metric]
        for metric in metric_names
    }
    for name, metrics in reaction_metrics.items()
}).T

reaction_results.index.name = 'Модель'

display(
    reaction_results.style
    .format('{:.4f}')
    .background_gradient(cmap='Greens', axis=0)
)

per_class_rows = []

for name, metrics in reaction_metrics.items():
    class_metrics = metrics['per_class_metrics']

    for label in REACTION_CLASSES:
        per_class_rows.append({
            'Модель': name,
            'Класс': label,
            'Support': int((y_hidden_reaction == label).sum()),
            'Precision': class_metrics[f'precision_{label}'],
            'Recall': class_metrics[f'recall_{label}'],
            'F1': class_metrics[f'f1_{label}'],
        })

per_class_results = pd.DataFrame(per_class_rows).set_index(
    ['Модель', 'Класс']
)

display(
    per_class_results.style.format({
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1': '{:.4f}',
    })
)

for name, metrics in reaction_metrics.items():
    print(f'\n{name}: матрица ошибок')

    display(pd.DataFrame(
        metrics['confusion_matrix'],
        index=pd.Index(REACTION_CLASSES, name='Истинная'),
        columns=pd.Index(REACTION_CLASSES, name='Предсказанная'),
    ))

Диалогов для оценки: 990

Распределение реакций:
negative    330
positive    330
neutral     330
Name: count, dtype: int64


/Users/var/Documents/projects/meow/HW/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DummyClassifier from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/var/Documents/projects/meow/HW/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/var/Documents/projects/meow/HW/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectori

,accuracy,balanced_accuracy,macro-f1,weighted-f1
Модель,,,,
dummy_most_frequent,0.3333,0.3333,0.1667,0.1667
dummy_stratified,0.3505,0.3505,0.3223,0.3223
complement_nb,0.3848,0.3848,0.3373,0.3373
log_reg,0.4505,0.4505,0.4473,0.4473
linear_svc,0.4354,0.4354,0.4060,0.4060
my_model,0.4869,0.4869,0.4863,0.4863



dummy_most_frequent: матрица ошибок


Предсказанная,negative,neutral,positive
Истинная,,,
negative,0,330,0
neutral,0,330,0
positive,0,330,0


...

Предсказанная,negative,neutral,positive
Истинная,,,
negative,180,136,14
neutral,150,170,10
positive,129,105,96



linear_svc: матрица ошибок


Предсказанная,negative,neutral,positive
Истинная,,,
negative,161,164,5
neutral,112,217,1
positive,119,158,53



my_model: матрица ошибок


Предсказанная,negative,neutral,positive
Истинная,,,
negative,187,134,9
neutral,136,184,10
positive,110,109,111


In [5]:
def show_reaction_errors(model_name, n=5, max_chars=1500):
    predictions = reaction_predictions[model_name]

    error_indices = [
        i for i, (true, pred) in enumerate(
            zip(y_hidden_reaction, predictions)
        )
        if true != pred
    ]

    # Получаем диалоги в том же порядке, что и предсказания реакций.
    dialogs_by_id = {
        d["dialog_id"]: d
        for d in read_dialogs("hidden_test.jsonl")
    }

    print(f"{model_name}: ошибок {len(error_indices)}")

    for i in error_indices[:n]:
        dialog = dialogs_by_id[hidden_ids[i]]

        print("\n" + "=" * 80)
        print("ID:", dialog["dialog_id"])
        print("Истинная реакция:", y_hidden_reaction[i])
        print("Предсказанная:", predictions[i])

        for turn in dialog["turns"]:
            text = turn["text"]
            if max_chars is not None:
                text = text[:max_chars]

            print(f"\n[{turn['role']}]\n{text}")

In [15]:
show_reaction_errors("my_model", n=5)

# show_reaction_errors("my_model", n=5)

my_model: ошибок 508

ID: t62f1ed47f1
Истинная реакция: negative
Предсказанная: neutral

[user]
What are the all (subject, relation, and object) triplets in each of the following sentences?
(1) 'A girl sitting on a bench reading, a book in one hand and half of an apple in her other hand, with her legs crossed, and another bench, grass, and a building behind her.'
(2) 'Three people and a black car are dressed in ski jackets and two are wearing skis and our out in the snow while the third person is off to the side with her snowboard.'
(3) 'A man, with other people in the background, is sitting at a picnic table with a cell phone in front of him while he uses a fork and knife to cut a dessert on a plate.'
(4) 'A tennis court shows a man almost against the net, double-gripping his racquet and tapping a tennis ball to a second player with a wide stance and an expectant look.'
(5) 'View from a few feet away of the silhouette of a person standing on a surfboard riding on a white wave in dark 

# Тест темы

In [7]:
hidden_dialogs = read_dialogs('hidden_test.jsonl')

with open('hidden_labels.jsonl', encoding='utf-8') as file:
    hidden_labels = {
        record['dialog_id']: record['topic']
        for line in file
        if line.strip()
        for record in [json.loads(line)]
    }

hidden_dialogs = [
    dialog
    for dialog in hidden_dialogs
    if dialog['dialog_id'] in hidden_labels
]

hidden_ids = [dialog['dialog_id'] for dialog in hidden_dialogs]

X_hidden = [
    [
        turn['text']
        for turn in dialog['turns']
        if turn['role'] == 'user'
    ]
    for dialog in hidden_dialogs
]

y_hidden = np.array([
    hidden_labels[dialog_id]
    for dialog_id in hidden_ids
])

print('Диалогов для оценки:', len(hidden_dialogs))
print('\nЭталонные темы:')
print(pd.Series(y_hidden).value_counts())

Диалогов для оценки: 990

Эталонные темы:
SOFTWARE AND WEB DEVELOPMENT            400
ENTERTAINMENT                            50
WRITING JOURNALISM AND PUBLISHING        50
LANGUAGE                                 49
TECHNOLOGY                               49
HEALTH AND MEDICINE                      49
MATH LOGIC AND STATISTICS                49
PHYSICS CHEMISTRY AND ASTRONOMY          49
EDUCATION                                49
GAMES                                    49
BUSINESS AND MARKETING                   49
CAREER AND JOB APPLICATION               49
AI MACHINE LEARNING AND DATA SCIENCE     49
Name: count, dtype: int64


In [8]:
def predict_saved_topic_model(artifact, user_texts):
    model = artifact['model']

    if isinstance(model, MyModelTopics):
        return model.predict(user_texts)

    if isinstance(model, Pipeline):
        documents = ['\n'.join(parts) for parts in user_texts]
        return model.predict(documents)

    X_tfidf, _ = vectorize_topic_texts(user_texts, vectorizer=artifact['vectorizer'])
    return model.predict(X_tfidf)

model_paths = sorted(Path('artifacts/topics').glob('topic_*.joblib'))

hidden_predictions = {}
saved_topic_names = {}

for path in model_paths:
    artifact = joblib.load(path)
    
    hidden_predictions[path.stem] = predict_saved_topic_model(artifact, X_hidden)
    saved_topic_names[path.stem] = artifact.get('topic_names', {})

    print('Получены предсказания:', path.stem)

Получены предсказания: topic_dummy
Получены предсказания: topic_knn
Получены предсказания: topic_logreg_best
Получены предсказания: topic_logreg_cv
Получены предсказания: topic_logreg_tfidf
Получены предсказания: topic_my_model_weight1
Получены предсказания: topic_svc


In [9]:
def calculate_topic_partition_metrics(y_true, y_pred):
    table = contingency_matrix(y_true, y_pred)

    purity = table.max(axis=0).sum() / table.sum()

    return {
        'ARI': adjusted_rand_score(y_true, y_pred),
        'NMI': normalized_mutual_info_score(y_true, y_pred),
        'Однородность': homogeneity_score(y_true, y_pred),
        'Полнота': completeness_score(y_true, y_pred),
        'V-мера': v_measure_score(y_true, y_pred),
        'Чистота': purity,
    }


hidden_metrics = pd.DataFrame({
    name: calculate_topic_partition_metrics(y_hidden, predictions)
    for name, predictions in hidden_predictions.items()
}).T

hidden_metrics.index.name = 'Модель'

display(hidden_metrics.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0))

hidden_metrics.to_csv('hidden_topic_metrics.csv', encoding='utf-8-sig')

,ARI,NMI,Однородность,Полнота,V-мера,Чистота
Модель,,,,,,
topic_dummy,0.0000,0.0000,0.0000,1.0000,0.0000,0.4040
topic_knn,0.3578,0.2843,0.2239,0.3892,0.2843,0.4657
topic_logreg_best,0.4549,0.3538,0.2940,0.4439,0.3538,0.5030
topic_logreg_cv,0.4549,0.3538,0.2940,0.4439,0.3538,0.5030
topic_logreg_tfidf,0.4549,0.3538,0.2940,0.4439,0.3538,0.5030
topic_my_model_weight1,0.3434,0.2655,0.2215,0.3311,0.2655,0.4616
topic_svc,0.3821,0.3215,0.2557,0.4330,0.3215,0.4768


In [10]:
def show_topic_correspondence(model_name):
    predictions = hidden_predictions[model_name]
    topic_names = saved_topic_names[model_name]

    predicted_names = [
        topic_names.get(
            int(label),
            topic_names.get(str(label), str(label)),
        )
        for label in predictions
    ]

    table = pd.crosstab(
        pd.Series(predicted_names, name='Предсказанная тема'),
        pd.Series(y_hidden, name='Эталонная тема'),
    )

    print('Количество диалогов:')
    display(table)

    print('Доли эталонных тем внутри каждого предсказанного класса:')
    proportions = table.div(table.sum(axis=1), axis=0)
    

    display(
        proportions.style
        .format('{:.1%}')
        .background_gradient(cmap='Blues', axis=None)
    )

show_topic_correspondence('topic_logreg_best')

Количество диалогов:


Эталонная тема,AI MACHINE LEARNING AND DATA SCIENCE,BUSINESS AND MARKETING,CAREER AND JOB APPLICATION,EDUCATION,ENTERTAINMENT,GAMES,HEALTH AND MEDICINE,LANGUAGE,MATH LOGIC AND STATISTICS,PHYSICS CHEMISTRY AND ASTRONOMY,SOFTWARE AND WEB DEVELOPMENT,TECHNOLOGY,WRITING JOURNALISM AND PUBLISHING
Предсказанная тема,,,,,,,,,,,,,
Вопросы про ИИ,7,0,0,0,2,0,0,0,0,0,0,4,0
"Математика, расчеты, решение задач",1,2,0,2,0,1,1,0,29,13,1,1,1
Объяснение и разъяснение,11,12,7,15,15,9,27,11,10,20,28,25,2
Перевод и работа с языками,0,0,0,1,1,0,1,25,2,0,0,0,0
Программирование,25,2,1,1,0,8,4,0,6,1,352,14,0
Прочее,0,0,1,0,2,10,1,1,0,0,3,0,0
Работа с текстом,5,33,40,30,30,21,15,12,2,15,16,5,47


Доли эталонных тем внутри каждого предсказанного класса:


Эталонная тема,AI MACHINE LEARNING AND DATA SCIENCE,BUSINESS AND MARKETING,CAREER AND JOB APPLICATION,EDUCATION,ENTERTAINMENT,GAMES,HEALTH AND MEDICINE,LANGUAGE,MATH LOGIC AND STATISTICS,PHYSICS CHEMISTRY AND ASTRONOMY,SOFTWARE AND WEB DEVELOPMENT,TECHNOLOGY,WRITING JOURNALISM AND PUBLISHING
Предсказанная тема,,,,,,,,,,,,,
Вопросы про ИИ,53.8%,0.0%,0.0%,0.0%,15.4%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,30.8%,0.0%
"Математика, расчеты, решение задач",1.9%,3.8%,0.0%,3.8%,0.0%,1.9%,1.9%,0.0%,55.8%,25.0%,1.9%,1.9%,1.9%
Объяснение и разъяснение,5.7%,6.2%,3.6%,7.8%,7.8%,4.7%,14.1%,5.7%,5.2%,10.4%,14.6%,13.0%,1.0%
Перевод и работа с языками,0.0%,0.0%,0.0%,3.3%,3.3%,0.0%,3.3%,83.3%,6.7%,0.0%,0.0%,0.0%,0.0%
Программирование,6.0%,0.5%,0.2%,0.2%,0.0%,1.9%,1.0%,0.0%,1.4%,0.2%,85.0%,3.4%,0.0%
Прочее,0.0%,0.0%,5.6%,0.0%,11.1%,55.6%,5.6%,5.6%,0.0%,0.0%,16.7%,0.0%,0.0%
Работа с текстом,1.8%,12.2%,14.8%,11.1%,11.1%,7.7%,5.5%,4.4%,0.7%,5.5%,5.9%,1.8%,17.3%


In [11]:
with open("hidden_labels.jsonl", encoding="utf-8") as file:
    topic_labels = {
        record["dialog_id"]: record["topic"]
        for line in file
        if line.strip()
        for record in [json.loads(line)]
        if record.get("topic") is not None
    }

topic_hidden_ids = [
    d["dialog_id"]
    for d in read_dialogs("hidden_test.jsonl")
    if d["dialog_id"] in topic_labels
]


def show_topic_examples(
    model_name,
    n=5,
    true_topic=None,
    predicted_class=None,
    max_chars=1500,
):
    predictions = hidden_predictions[model_name]
    names = saved_topic_names[model_name]

    indices = [
        i for i, (true, pred) in enumerate(zip(y_hidden, predictions))
        if (true_topic is None or true == true_topic)
        and (
            predicted_class is None
            or str(pred) == str(predicted_class)
        )
    ]

    print(f"Подходящих примеров: {len(indices)}")

    for i in indices[:n]:
        pred = predictions[i]
        topic_name = names.get(
            int(pred),
            names.get(str(pred), str(pred)),
        )

        print("\n" + "=" * 80)
        print("ID:", topic_hidden_ids[i])
        print("Эталонная тема:", y_hidden[i])
        print(f"Предсказанная: {pred} — {topic_name}")

        for j, text in enumerate(X_hidden[i], start=1):
            if max_chars is not None:
                text = text[:max_chars]

            print(f"\n[User, реплика {j}]\n{text}")

In [16]:
# show_topic_examples("topic_logreg_best", n=5)

show_topic_examples(
    "topic_logreg_best",
    true_topic="EDUCATION",
    n=5,
)

show_topic_examples(
    "topic_logreg_best",
    true_topic="LANGUAGE",
    predicted_class=4,
    n=5,
)

Подходящих примеров: 49

ID: t261b98f77c
Эталонная тема: EDUCATION
Предсказанная: 3 — Работа с текстом

[User, реплика 1]
Introduction: The issue of climate cooperation between the U.S. and China in the 21st century presents complex challenges and opportunities for collaboration in addressing pollution. Drawing insights from The Odyssey, we can better understand the evolving dynamics of this important bilateral relationship as it pertains to environmental stewardship. The narrative of Odysseus facing obstacles and forging alliances can be paralleled with the contemporary struggles and agreements between nations on pollution control. This research project aims to explore the implications of climate cooperation between the U.S. and China through the lens of The Odyssey, providing a unique perspective on this critical issue of combating pollution.


 Body Paragraph 1: One aspect of the issue revolves around the environmental policies and practices of both the U.S. and China in addressing 